In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Anggaran penggunaan: 7 minit pada pemproses Heron r2 (NOTA: Ini adalah anggaran sahaja. Masa larian anda mungkin berbeza.)*

##  Hasil pembelajaran

Kami mencadangkan agar pengguna biasa dengan topik berikut sebelum melalui tutorial ini:
- Asas dynamical decoupling, mitigasi ralat pengukuran, gate twirling, dan zero-noise extrapolation, seperti yang diterangkan dalam [panduan](/guides/error-mitigation-and-suppression-techniques) ini.

## Prasyarat

Setelah melalui tutorial ini, pengguna seharusnya memahami:

- Cara teknik mitigasi ralat yang disebutkan di atas dilaksanakan secara selektif pada perkakasan.
- Cara ia dibandingkan dari segi keupayaan untuk mengurangkan bunyi perkakasan.

## Latar Belakang

Tutorial ini meneroka pilihan penindasan ralat dan mitigasi ralat yang tersedia dengan primitif Estimator daripada Qiskit Runtime. Tutorial ini menunjukkan cara melaksanakan setiap kaedah berikut secara individu:

- Dynamical decoupling
- Mitigasi ralat pengukuran
- Gate twirling
- Zero-noise extrapolation (ZNE)

Perlu diingat bahawa alternatif kepada melaksanakan teknik ini secara individu ialah melaksanakannya menggunakan [tahap ketahanan (resilience level)](/guides/estimator-noise-management), di mana `resilience_level` mengambil nilai 0, 1, 2:

- 0 : Tiada mitigasi dilaksanakan.
- 1 : Mitigasi ralat pengukuran dilaksanakan.
- 2 : Gate twirling, mitigasi ralat pengukuran, dan ZNE dilaksanakan.

Dalam tutorial ini, anda akan membina Circuit dan observable, kemudian menghantar kerja menggunakan primitif Estimator dengan kombinasi tetapan mitigasi ralat yang berbeza. Selepas itu, anda akan memplot hasilnya untuk melihat kesan pelbagai tetapan tersebut. Sebahagian besar tutorial menggunakan Circuit 10-Qubit untuk memudahkan visualisasi, dan pada penghujung, anda akan skala naik aliran kerja kepada 50 qubit.

## Keperluan

Sebelum memulakan panduan ini, pastikan anda telah memasang yang berikut:

- Qiskit SDK v2.1 atau lebih baharu, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.40 atau lebih baharu (`pip install qiskit-ibm-runtime`)

## Persediaan

In [7]:
import matplotlib.pyplot as plt
import numpy as np

from qiskit.circuit.library import efficient_su2, unitary_overlap
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Batch, EstimatorV2 as Estimator

## Contoh simulator berskala kecil
Kami akan melewatkan langkah ini kerana mitigasi ralat runtime tidak disokong pada simulator.
## Contoh perkakasan
### Langkah 1: Petakan input klasik kepada masalah kuantum
Panduan ini mengandaikan bahawa masalah klasik sudah dipetakan kepada kuantum. Mulakan dengan membina Circuit dan observable untuk diukur. Walaupun teknik yang digunakan di sini boleh diaplikasi kepada pelbagai jenis Circuit, untuk kesederhanaan panduan ini menggunakan Circuit [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) yang disertakan dalam pustaka Circuit Qiskit.

`efficient_su2` ialah Circuit kuantum berparameter yang direka untuk dilaksanakan dengan cekap pada perkakasan kuantum dengan sambungan qubit yang terhad, namun masih cukup ekspresif untuk menyelesaikan masalah dalam domain aplikasi seperti pengoptimuman dan kimia. Ia dibina dengan melapisi lapisan gate satu-Qubit berparameter secara bergilir dengan lapisan yang mengandungi corak tetap gate dua-Qubit, untuk bilangan pengulangan yang dipilih. Corak gate dua-Qubit boleh ditentukan oleh pengguna. Di sini anda boleh menggunakan corak bawaan `pairwise` kerana ia meminimumkan kedalaman Circuit dengan mengemas gate dua-Qubit sepadat mungkin. Corak ini boleh dilaksanakan menggunakan sambungan qubit linear sahaja.

In [8]:
n_qubits = 10
reps = 1

circuit = efficient_su2(n_qubits, entanglement="pairwise", reps=reps)

circuit.decompose().draw("mpl", scale=0.7)

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/24abd7ba-bbb8-443b-9e81-866795d39a6c-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/24abd7ba-bbb8-443b-9e81-866795d39a6c-0.avif)

Sebagai observable kita, mari ambil operator Pauli $Z$ yang bertindak pada qubit terakhir, $Z I \cdots I$.
Perlu diingat bahawa fakta bahawa qubit terakhir sepadan dengan elemen pertama rentetan ini adalah disebabkan oleh penggunaan notasi little-endian oleh Qiskit.

In [9]:
# Z on the last qubit (index -1) with coefficient 1.0
observable = SparsePauliOp.from_sparse_list(
    [("Z", [-1], 1.0)], num_qubits=n_qubits
)

Pada ketika ini, anda boleh teruskan dengan menjalankan Circuit dan mengukur observable. Walau bagaimanapun, anda juga mahu membandingkan output peranti kuantum dengan jawapan yang betul — iaitu, nilai teori observable, sekiranya Circuit dilaksanakan tanpa ralat. Untuk Circuit kuantum kecil, anda boleh mengira nilai ini dengan mensimulasikan Circuit pada komputer klasik, tetapi ini tidak mungkin untuk Circuit yang lebih besar, berskala utiliti. Anda boleh mengatasi isu ini dengan teknik "mirror circuit" (juga dikenali sebagai "compute-uncompute"), yang berguna untuk menanda aras prestasi peranti kuantum.

#### Circuit cermin
Dalam teknik Circuit cermin, anda menggabungkan Circuit dengan Circuit songsang (inverse circuit)-nya, yang dibentuk dengan menyongsangkan setiap gate Circuit dalam susunan terbalik. Circuit yang terhasil melaksanakan operator identiti, yang boleh disimulasikan dengan mudah. Kerana struktur Circuit asal dipelihara dalam Circuit cermin, melaksanakan Circuit cermin tetap memberikan gambaran tentang bagaimana peranti kuantum akan berprestasi pada Circuit asal.

Sel kod berikut menetapkan parameter rawak kepada Circuit anda, kemudian membina Circuit cermin menggunakan kelas [`unitary_overlap`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.unitary_overlap). Sebelum mencerminkan Circuit, tambahkan arahan [barrier](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Barrier) padanya untuk menghalang Transpiler daripada menggabungkan dua bahagian Circuit di kedua-dua sisi barrier dan menghasilkan Circuit yang telah ditransformasi tanpa sebarang gate.

In [10]:
# Generate random parameters
rng = np.random.default_rng(1234)
params = rng.uniform(-np.pi, np.pi, size=circuit.num_parameters)

# Assign the parameters to the circuit
assigned_circuit = circuit.assign_parameters(params)

# Add a barrier to prevent circuit optimization of mirrored operators
assigned_circuit.barrier()

# Construct mirror circuit
mirror_circuit = unitary_overlap(assigned_circuit, assigned_circuit)

mirror_circuit.decompose().draw("mpl", scale=0.7)

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/4dbde811-1ba9-47a8-85a0-dcaff054ed60-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/4dbde811-1ba9-47a8-85a0-dcaff054ed60-0.avif)

### Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Anda mesti mengoptimumkan Circuit anda sebelum menjalankannya pada perkakasan. Proses ini melibatkan beberapa langkah:

- Pilih susun atur qubit yang memetakan qubit maya Circuit anda kepada qubit fizikal pada perkakasan.
- Sisipkan gate swap mengikut keperluan untuk menghala interaksi antara qubit yang tidak bersambung.
- Terjemahkan gate dalam Circuit anda kepada arahan [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) yang boleh dilaksanakan terus pada perkakasan.
- Lakukan pengoptimuman Circuit untuk meminimumkan kedalaman Circuit dan bilangan gate.

Transpiler yang dibina dalam Qiskit boleh melakukan semua langkah ini untuk anda. Kerana contoh ini menggunakan Circuit yang cekap dari segi perkakasan, Transpiler sepatutnya dapat memilih susun atur qubit yang tidak memerlukan sebarang gate swap disisipkan untuk menghala interaksi.

Anda perlu memilih peranti perkakasan yang hendak digunakan sebelum mengoptimumkan Circuit anda. Sel kod berikut meminta peranti yang paling-tidak-sibuk dengan sekurang-kurangnya 127 qubit.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

In [12]:
print(backend)

<IBMBackend('ibm_fez')>


You can transpile your circuit to your chosen backend by creating a pass manager and then running the pass manager on the circuit. An easy way to create a pass manager is to use the [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) function. See [Transpile with pass managers](/docs/guides/transpile-with-pass-managers) for a more detailed explanation of transpiling with pass managers.

In [13]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=1234
)
isa_circuit = pass_manager.run(mirror_circuit)

isa_circuit.draw("mpl", idle_wires=False, scale=0.7, fold=-1)

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/027e829a-44d3-455e-b2bf-8ce0d7e26b9b-0.avif" alt="Output of the previous code cell" />

Anda boleh mentransformasi Circuit anda kepada Backend yang dipilih dengan membuat pengurus laluan kemudian menjalankan pengurus laluan pada Circuit. Cara mudah untuk membuat pengurus laluan ialah menggunakan fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager). Lihat [Transpile with pass managers](/guides/transpile-with-pass-managers) untuk penjelasan lebih terperinci tentang transformasi dengan pengurus laluan.

In [14]:
isa_observable = observable.apply_layout(isa_circuit.layout)

print("Original observable:")
print(observable)
print()
print("Observable with layout applied:")
print(isa_observable)

Original observable:
SparsePauliOp(['ZIIIIIIIII'],
              coeffs=[1.+0.j])

Observable with layout applied:
SparsePauliOp(['IIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII'],
              coeffs=[1.+0.j])


![Output of the previous code cell](../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/027e829a-44d3-455e-b2bf-8ce0d7e26b9b-0.avif)

Circuit yang telah ditransformasi kini mengandungi hanya arahan ISA. Semua gate telah didekomposisi dalam bentuk gate $\sqrt{X}$ dan putaran $R_z$, dan [gate CZ](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.CZGate).

Proses transformasi telah memetakan qubit maya Circuit kepada qubit fizikal pada perkakasan. Maklumat tentang susun atur qubit disimpan dalam atribut `layout` Circuit yang telah ditransformasi. Observable juga ditakrifkan dari segi qubit maya, jadi anda perlu menggunakan susun atur ini pada observable, yang boleh anda lakukan dengan kaedah [`apply_layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp#apply_layout) bagi `SparsePauliOp`.

In [ ]:
pub = (isa_circuit, isa_observable)

jobs = []

with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)
    estimator.options.environment.job_tags = [
        "TUT_CEM_SS"
    ]  # add tag for this small scale job
    # Set number of shots
    estimator.options.default_shots = 100_000
    # Disable runtime compilation and error mitigation
    estimator.options.resilience_level = 0

    # Run job with no error mitigation
    job0 = estimator.run([pub])
    jobs.append(job0)

    # Add dynamical decoupling (DD)
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XpXm"
    job1 = estimator.run([pub])
    jobs.append(job1)

    # Add readout error mitigation (DD + TREX)
    estimator.options.resilience.measure_mitigation = True
    job2 = estimator.run([pub])
    jobs.append(job2)

    # Add gate twirling (DD + TREX + Gate Twirling)
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    job3 = estimator.run([pub])
    jobs.append(job3)

    # Add zero-noise extrapolation (DD + TREX + Gate Twirling + ZNE)
    estimator.options.resilience.zne_mitigation = True
    estimator.options.resilience.zne.noise_factors = (1, 3, 5)
    estimator.options.resilience.zne.extrapolator = ("exponential", "linear")
    job4 = estimator.run([pub])
    jobs.append(job4)

### Step 4: Post-process and return result in desired classical format

Finally, you can analyze the data. Here you will retrieve the job results, extract the measured expectation values from them, and plot the values, including error bars of one standard deviation.

In [16]:
# Retrieve the job results
results = [job.result() for job in jobs]

# Unpack the PUB results (there's only one PUB result in each job result)
pub_results = [result[0] for result in results]

# Unpack the expectation values and standard errors
expectation_vals = np.array(
    [float(pub_result.data.evs) for pub_result in pub_results]
)
standard_errors = np.array(
    [float(pub_result.data.stds) for pub_result in pub_results]
)

# Plot the expectation values
fig, ax = plt.subplots()
labels = ["No mitigation", "+ DD", "+ TREX", "+ Twirling", "+ ZNE"]
ax.bar(
    range(len(labels)),
    expectation_vals,
    yerr=standard_errors,
    label="experiment",
)
ax.axhline(y=1.0, color="gray", linestyle="--", label="ideal")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Expectation value")
ax.legend(loc="upper left")

plt.show()

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/eef38976-0ca2-429a-b2dc-41aac69605f7-0.avif" alt="Output of the previous code cell" />

### Langkah 3: Jalankan menggunakan primitif Qiskit
Anda kini sedia untuk menjalankan Circuit menggunakan primitif Estimator.

Di sini anda akan menghantar lima kerja berasingan, bermula tanpa penindasan atau mitigasi ralat, kemudian menghidupkan pelbagai pilihan penindasan dan mitigasi ralat secara berurutan yang tersedia dalam Qiskit Runtime. Untuk maklumat tentang pilihan tersebut, rujuk halaman berikut:

- [Gambaran keseluruhan semua pilihan](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options)
- [Dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options)
- [Resilience, termasuk mitigasi ralat pengukuran dan zero-noise extrapolation (ZNE)](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2)
- [Twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)

Kerana kerja-kerja ini boleh dijalankan secara bebas antara satu sama lain, anda boleh menggunakan [mod batch](/guides/run-jobs-batch) untuk membolehkan Qiskit Runtime mengoptimumkan masa pelaksanaannya.

In [17]:
n_qubits = 50
reps = 1

# Construct circuit and observable
circuit = efficient_su2(n_qubits, entanglement="pairwise", reps=reps)
observable = SparsePauliOp.from_sparse_list(
    [("Z", [-1], 1.0)], num_qubits=n_qubits
)

# Assign parameters to circuit
params = rng.uniform(-np.pi, np.pi, size=circuit.num_parameters)
assigned_circuit = circuit.assign_parameters(params)
assigned_circuit.barrier()

# Construct mirror circuit
mirror_circuit = unitary_overlap(assigned_circuit, assigned_circuit)

# Transpile circuit and observable
isa_circuit = pass_manager.run(mirror_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Run jobs
pub = (isa_circuit, isa_observable)

jobs = []

with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)
    estimator.options.environment.job_tags = [
        "TUT_CEM_LS"
    ]  # add tag for this large scale job
    # Set number of shots
    estimator.options.default_shots = 100_000
    # Disable runtime compilation and error mitigation
    estimator.options.resilience_level = 0

    # Run job with no error mitigation
    job0 = estimator.run([pub])
    jobs.append(job0)

    # Add dynamical decoupling (DD)
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XpXm"
    job1 = estimator.run([pub])
    jobs.append(job1)

    # Add readout error mitigation (DD + TREX)
    estimator.options.resilience.measure_mitigation = True
    job2 = estimator.run([pub])
    jobs.append(job2)

    # Add gate twirling (DD + TREX + Gate Twirling)
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    job3 = estimator.run([pub])
    jobs.append(job3)

    # Add zero-noise extrapolation (DD + TREX + Gate Twirling + ZNE)
    estimator.options.resilience.zne_mitigation = True
    estimator.options.resilience.zne.noise_factors = (1, 3, 5)
    estimator.options.resilience.zne.extrapolator = ("exponential", "linear")
    job4 = estimator.run([pub])
    jobs.append(job4)

# Retrieve the job results
results = [job.result() for job in jobs]

# Unpack the PUB results (there's only one PUB result in each job result)
pub_results = [result[0] for result in results]

# Unpack the expectation values and standard errors
expectation_vals = np.array(
    [float(pub_result.data.evs) for pub_result in pub_results]
)
standard_errors = np.array(
    [float(pub_result.data.stds) for pub_result in pub_results]
)

# Plot the expectation values
fig, ax = plt.subplots()
labels = ["No mitigation", "+ DD", "+ TREX", "+ Twirling", "+ ZNE"]
ax.bar(
    range(len(labels)),
    expectation_vals,
    yerr=standard_errors,
    label="experiment",
)
ax.axhline(y=1.0, color="gray", linestyle="--", label="ideal")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Expectation value")
ax.legend(loc="upper left")

plt.show()

<Image src="../docs/images/tutorials/combine-error-mitigation-techniques/extracted-outputs/d7d8408b-faf1-4eda-ab9c-bdeaab01ff53-0.avif" alt="Output of the previous code cell" />

### Langkah 4: Proses pasca dan kembalikan hasil dalam format klasik yang dikehendaki
Akhir sekali, anda boleh menganalisis data. Di sini anda akan mendapatkan semula keputusan kerja, mengekstrak nilai jangkaan yang diukur daripadanya, dan memplot nilai tersebut, termasuk bar ralat satu sisihan piawai.